<a href="https://colab.research.google.com/github/safira1104/Cashed/blob/main/Normalizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# CELL 1 - INSTALL AMAN UNTUK NLP
# ==========================================

# 1. Copot paksa library gambar dan audio bawaan Colab agar tidak bentrok
!pip uninstall torchvision torchaudio -y

# 2. Install library NLP (TANPA FLAG -U)
!pip install transformers datasets accelerate

import torch
import pandas as pd
import re

from datasets import Dataset, DatasetDict
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PyTorch Version: {torch.__version__}")
print(f"Device yang digunakan: {device.upper()}")

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
PyTorch Version: 2.11.0+cu128
Device yang digunakan: CUDA


In [2]:
# ==========================================
# CELL 2 - PARSING SEMUA FILE .TEXTPROTO
# ==========================================
print("Mengekstrak kalimat dari file .textproto...")

def parse_textproto_to_sentences(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
    except Exception as e:
        print(f"Gagal membaca {file_path}: {e}")
        return pd.DataFrame(columns=["kotor", "bersih"])

    sentence_blocks = content.split('sentences {')
    data = []

    for block in sentence_blocks[1:]:
        kotor_tokens = []
        bersih_tokens = []
        token_blocks = block.split('tokens {')

        for token in token_blocks[1:]:
            expanded_match = re.search(r'expanded:\s*"([^"]+)"', token)
            abbrev_match = re.search(r'abbreviated:\s*"([^"]+)"', token)

            if expanded_match:
                expanded_word = expanded_match.group(1)

                if abbrev_match:
                    kotor_tokens.append(abbrev_match.group(1))
                else:
                    kotor_tokens.append(expanded_word)

                bersih_tokens.append(expanded_word)

        if kotor_tokens and bersih_tokens:
            kalimat_kotor = " ".join(kotor_tokens)
            kalimat_bersih = " ".join(bersih_tokens)
            kalimat_kotor = re.sub(r'\s+([?.!,:;])', r'\1', kalimat_kotor)
            kalimat_bersih = re.sub(r'\s+([?.!,:;])', r'\1', kalimat_bersih)

            if kalimat_kotor != kalimat_bersih:
                data.append({"kotor": kalimat_kotor, "bersih": kalimat_bersih})

    return pd.DataFrame(data)

# Proses ketiga dataset
df_train = parse_textproto_to_sentences("/content/sample_data/train.textproto")
df_dev = parse_textproto_to_sentences("/content/sample_data/dev.textproto")
df_test = parse_textproto_to_sentences("/content/sample_data/test.textproto")

print(f"Jumlah data Train : {len(df_train)}")
print(f"Jumlah data Dev   : {len(df_dev)}")
print(f"Jumlah data Test  : {len(df_test)}")

Mengekstrak kalimat dari file .textproto...
Jumlah data Train : 21164
Jumlah data Dev   : 2648
Jumlah data Test  : 2655


In [3]:
# ==========================================
# CELL 3 - PREPROCESSING & TOKENISASI
# ==========================================
# Tambahkan import di sini agar aman

print("Memuat Tokenizer T5-Small...")
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Menggabungkan data Pandas ke format Hugging Face
hf_dataset = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "validation": Dataset.from_pandas(df_dev),
    "test": Dataset.from_pandas(df_test)
})

prefix = "normalize: "

def preprocess_function(examples):
    # Input teks kotor
    inputs = [prefix + doc for doc in examples["kotor"]]
    # Target teks bersih
    targets = [doc for doc in examples["bersih"]]

    # Tokenisasi Input
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")
    # Tokenisasi Target
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Mulai proses tokenisasi dataset (Ini mungkin memakan waktu sebentar)...")
tokenized_datasets = hf_dataset.map(preprocess_function, batched=True)
print("Tokenisasi selesai!")

Memuat Tokenizer T5-Small...


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Mulai proses tokenisasi dataset (Ini mungkin memakan waktu sebentar)...


Map:   0%|          | 0/21164 [00:00<?, ? examples/s]

Map:   0%|          | 0/2648 [00:00<?, ? examples/s]

Map:   0%|          | 0/2655 [00:00<?, ? examples/s]

Tokenisasi selesai!


In [6]:
# ==========================================
# CELL 4 - TRAINING (FINE-TUNING) MODEL
# ==========================================

# Pastikan device terdefinisi ulang untuk menghindari error
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "t5-small"

print("Memuat Base Model T5-Small...")
model = T5ForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

print("Mengatur Konfigurasi Training...")
training_args = TrainingArguments(
    output_dir="./t5_normalizer_checkpoints",
    eval_strategy="epoch",            # <--- INI YANG DIPERBAIKI
    learning_rate=2e-4,
    per_device_train_batch_size=16,   # Gunakan 16 jika memori GPU cukup
    per_device_eval_batch_size=16,
    num_train_epochs=4,               # Latih selama 4 putaran (epochs)
    weight_decay=0.01,
    save_total_limit=2,               # Hanya simpan 2 checkpoint terakhir agar storage tidak penuh
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Menggunakan dev.textproto untuk validasi
)

print("MEMULAI PROSES TRAINING...")
trainer.train()

# ------------------------------------------
# PENYIMPANAN MODEL HASIL LATIHAN
# ------------------------------------------
folder_simpan = "./model_normalizer_saya_final"
trainer.save_model(folder_simpan)
tokenizer.save_pretrained(folder_simpan)

print(f"\nYEAY! Model berhasil dilatih dan disimpan di folder: {folder_simpan}")
print("Sekarang model ini bisa dipakai untuk mendeteksi singkatan otomatis!")

Memuat Base Model T5-Small...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Mengatur Konfigurasi Training...
MEMULAI PROSES TRAINING...


Epoch,Training Loss,Validation Loss
1,0.114465,0.081218
2,0.084645,0.063423
3,0.070774,0.056336
4,0.066736,0.054258


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


YEAY! Model berhasil dilatih dan disimpan di folder: ./model_normalizer_saya_final
Sekarang model ini bisa dipakai untuk mendeteksi singkatan otomatis!


In [7]:
# ==========================================
# CELL 5 - UJI COBA MODEL (INFERENCE)
# ==========================================
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# Pastikan device terdefinisi ulang
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Memuat Model Buatan Sendiri...")
# Memanggil dari folder lokal yang baru saja kita simpan
my_tokenizer = T5Tokenizer.from_pretrained("./model_normalizer_saya_final")
my_model = T5ForConditionalGeneration.from_pretrained("./model_normalizer_saya_final").to(device)

def bersihkan_kalimat(teks):
    teks_input = "normalize: " + teks
    inputs = my_tokenizer(teks_input, return_tensors="pt", max_length=128, truncation=True).to(device)

    with torch.no_grad():
        outputs = my_model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)

    return my_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Mari kita uji
test_sentences = [
    "the yelow weapons r suitable for dealing rapid damage.",
    "he is upset that he cant update his status",
    "at th turnamnt teams hd to present smthng based n ther projct."
]

print("\n--- HASIL UJI COBA MODEL BARU ---")
for kalimat in test_sentences:
    print(f"Input (Kotor)  : {kalimat}")
    print(f"Output (Bersih): {bersihkan_kalimat(kalimat)}\n")

Memuat Model Buatan Sendiri...


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


--- HASIL UJI COBA MODEL BARU ---
Input (Kotor)  : the yelow weapons r suitable for dealing rapid damage.
Output (Bersih): the following weapons are suitable for dealing rapid damage.

Input (Kotor)  : he is upset that he cant update his status
Output (Bersih): he is upset that he cannot update his status.

Input (Kotor)  : at th turnamnt teams hd to present smthng based n ther projct.
Output (Bersih): at the tournament teams had to present some based in their project.



In [8]:
from google.colab import drive
import shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Tentukan lokasi folder tujuan di Drive kamu
tujuan_drive = '/content/drive/MyDrive/Model_Normalizer_T5'

# 3. Copy folder model ke Drive
shutil.copytree('./model_normalizer_saya_final', tujuan_drive)

print("Folder model sudah tersimpan di Google Drive kamu!")

Mounted at /content/drive
Folder model sudah tersimpan di Google Drive kamu!
